**Step 1: Literature Input**

In [1]:
import pandas as pd
file_path = "articles.csv"

df = pd.read_csv(file_path, encoding="latin1")

print(df.head())
print("Total de artigos:", len(df))

         key                                              title  year  \
0  398100026  Methane yield enhancement of mesophilic and th...  2020   
1  398100027  Improved efficiency of anaerobic digestion thr...  2018   
2  398100028  Synergetic effects of anaerobic co-digestion o...  2023   
3  398100029  Unveiling microbial structures during raw micr...  2020   
4  398100032  Biomethanation and microbial community respons...  2021   

                            journal  \
0            Bioresource Technology   
1      Chemical Engineering Journal   
2            Bioresource Technology   
3  Science of The Total Environment   
4           Environmental Pollution   

                                             authors  \
0  Zhang, Le and Li, Fanghua and Kuroki, AgnÃÂ¨s...   
1  Lin, Richen and Cheng, Jun and Ding, Lingkan a...   
2  Khanthong, Kamonwan and Kadam, Rahul and Kim, ...   
3  Zamorano-LÃÂ³pez, N. and BorrÃÂ¡s, L. and Se...   
4  Ali, Gohar and Ling, Zhenmin and Saif, Irfa

In [2]:
bibliotecas = [
    'pandas as pd',
    'numpy as np',
    'matplotlib.pyplot as plt',
    'networkx as nx', 
    'nekomata'  
]

for lib in bibliotecas:
    try:
        exec(f"import {lib}")
        print(f"{lib} Installed successfully!")
    except ImportError as e:
        print(f" Error in{lib}: {e}")
        print("   Install with: pip install " + lib.split(' as ')[0])


pandas as pd Installed successfully!
numpy as np Installed successfully!
matplotlib.pyplot as plt Installed successfully!
networkx as nx Installed successfully!
 Error innekomata: No module named 'nekomata'
   Install with: pip install nekomata


pip install nekomata
pip install -r https://raw.githubusercontent.com/sysbio-curie/Neko/main/requirements.txt


In [3]:
import pandas as pd
df = pd.read_csv("articles.csv", encoding="latin1")
print("Columns:", df.columns.tolist())
print("\nShape:", df.shape)
print("\Sample:")
print(df[['title', 'authors', 'year', 'abstract']].head())
print("\nCommon keywords in keywords:")
if 'keywords' in df.columns:
    print(df['keywords'].dropna().str.split(', ').explode().value_counts().head(10))


Columns: ['key', 'title', 'year', 'journal', 'authors', 'abstract', 'doi', 'keywords']

Shape: (992, 8)
\Sample:
                                               title  \
0  Methane yield enhancement of mesophilic and th...   
1  Improved efficiency of anaerobic digestion thr...   
2  Synergetic effects of anaerobic co-digestion o...   
3  Unveiling microbial structures during raw micr...   
4  Biomethanation and microbial community respons...   

                                             authors  year  \
0  Zhang, Le and Li, Fanghua and Kuroki, AgnÃÂ¨s...  2020   
1  Lin, Richen and Cheng, Jun and Ding, Lingkan a...  2018   
2  Khanthong, Kamonwan and Kadam, Rahul and Kim, ...  2023   
3  Zamorano-LÃÂ³pez, N. and BorrÃÂ¡s, L. and Se...  2020   
4  Ali, Gohar and Ling, Zhenmin and Saif, Irfan a...  2021   

                                            abstract  
0  The impact of algal biochar addition on mesoph...  
1  Direct interspecies electron transfer (DIET) i...  
2  Anaerobic

In [4]:
import pandas as pd
import re
import networkx as nx  # For preview

# Load data
df = pd.read_csv("articles.csv", encoding="latin1")
print(f"Total articles: {len(df)}")

# Exact terms list (yours + common algae)
algae_terms = ['chlorella', 'scenedesmus', 'spirulina', 'chlamydomonas', 'cyanobacteria', 'microalgae']
reactor_terms = ['pbr', 'photobioreactor', 'hrap', 'high rate algal pond', 'cstr', 'uasb', 'anmbr']
process_terms = ['anaerobic digestion', 'methane', 'biogas', 'nutrient removal', 'phosphorous', 'nitrogen', 'ammonia', 'wastewater']
other_terms = ['melissa', 'celss', 'mes', 'mec', 'bes', 'immobilization', 'biofuel', 'bioremediation']

all_terms = algae_terms + reactor_terms + process_terms + other_terms
pattern = '|'.join([re.escape(t.lower()) for t in all_terms])

# Filter relevant articles
mask = (
    df['title'].str.contains(pattern, case=False, na=False) |
    df['abstract'].str.contains(pattern, case=False, na=False) |
    df['keywords'].str.contains(pattern, case=False, na=False)
)
df_filtered = df[mask].copy()
print(f"Relevant articles: {len(df_filtered)}")

# Detect species: binomial OR genus-like (RAW STRINGS!)
def extract_species(text):
    if pd.isna(text):
        return []
    # Genus exact algae OR short binomial
    alga_genera = r'\b(Chlorella|Scenedesmus|Spirulina|Chlamydomonas|Nannochloropsis|Botryococcus|Haematococcus|Tetradesmus)\b'
    binomial_short = r'\b[A-Z][a-z]{2,8}\s+[a-z]{3,10}\b'
    candidates = re.findall(alga_genera, text, re.I) + re.findall(binomial_short, text)
    stop_short = ['This study', 'The results', 'The present', 'This work']
    filtered = [c for c in candidates if c not in stop_short and len(c) > 3]
    return list(set(filtered))[:3]  # Unique, top 3


df_filtered['species'] = (
    df_filtered['title'].apply(extract_species) + 
    df_filtered['abstract'].apply(extract_species)
)
all_species = [sp for sublist in df_filtered['species'] for sp in sublist]
print("Clean top species:", pd.Series(all_species).value_counts().head(10))


df_filtered.to_csv("articles_filtered.csv", index=False)
print("Saved: articles_filtered.csv (use as input in your Ollama code)")


keywords_flat = df_filtered['keywords'].dropna().str.split(', ').explode()
print("Top keywords:", keywords_flat.value_counts().head(10))


Total articles: 992
Relevant articles: 974
Clean top species: Chlorella              393
Chlorella vulgaris     292
Scenedesmus             50
This review             29
The highest             29
The aim                 26
Anaerobic digestion     26
This paper              25
The use                 25
The study               24
Name: count, dtype: int64
Saved: articles_filtered.csv (use as input in your Ollama code)
Top keywords: keywords
Biofuel;Microalgae;Biodiesel;Bioethanol;Renewable energy                                                                                             2
Biochar amendment;Thermophilic anaerobic digestion;Mesophilic anaerobic digestion;Microbial enrichment;Methanogenic pathways                         1
Wastewater;Microalgae;Resource recovery;Bioenergy;Nanoparticles;Circular economy                                                                     1
Thiocyanate wastewater;AlgalÃ¢â¬âbacterial mixed culture;Lipid productivity;Photoautotrophic/photoh

**Step 2: Entity and relationship extraction with LLM**

In [5]:
import pandas as pd
import re
from openai import OpenAI
from tqdm.auto import tqdm

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

TAXA_KEYWORDS = [
    "Chlamydomonas reinhardtii",
    "Chlorella sp",
    "Chlorella fusca",
    "Chlorella minutissima",
    "Chlorella protothecoides",
    "Chlorella pyrenoidosa",
    "Chlorella sorokiniana",
    "Chlorella variabilis",
    "Chlorella vulgaris",
    "Dunaliella tertiolecta",
    "Tetradesmus obliquus",
    "Nannochloropsis gaditana",
    "Nannochloropsis salina",
    "Parachlorella kessleri",
    "Scenedesmus abundans",
    "Scenedesmus sp",
    "Scenedesmus obliquus",
    "Tetraselmis sp",
    "Tetraselmis elliptica",
    "Tetraselmis suecica",
    "Methanosarcina",
    "Arthrospira maxima",
    "Arthrospira sp",
    "Phormidium sp",
    "Pseudoscillatoria coralii",
    "Acutodesmus obliquus",
    "Chlamydomonas sp",
    "Chlorella kessleri",
    "Dunaliella salina",
    "Phaeodactylum tricornutum",
    "Scenedesmus obtusiusculus",
    "Methanosarcina acetivorans",
    "Methanosaeta",
    "Chlamydomonas acidophila",
    "Neochloris oleoabundans",
    "Spirulina sp",
    "Piscicoccus intestinalis",
    "Leptolyngbya angustata",
    "Synechococcus elongatus"
]

system_prompt = """
You extract ONLY species-to-outcome relations from scientific abstracts.

TARGET OUTCOMES:
1. nutrient removal
2. methane/biogas
3. biomass growth/productivity

RULES:
- Use ONLY taxa from the provided candidate taxa list.
- Extract only if the abstract states or clearly implies a relation between a taxon and one target outcome.
- Normalize all outcomes strictly as:
  nutrient removal
  methane/biogas
  biomass growth/productivity
- Valid evidence includes:
  nitrogen removal, phosphorus removal, ammonium removal, COD removal;
  methane production, methane yield, biogas production, methanogenesis enhancement;
  biomass growth, cell growth, biomass productivity, biomass yield.
- Keep taxon names exactly as provided in candidate taxa list.
- If no relation is present, return: ()
- No explanations.

STRICT FORMAT:
(taxon -> nutrient removal), (taxon -> methane/biogas), (taxon -> biomass growth/productivity)
Max 8 relations.
"""

def find_candidate_taxa(text, taxa_list):
    text_low = str(text).lower()
    matches = []
    for taxon in taxa_list:
        pattern = r'(?<!\w)' + re.escape(taxon.lower()) + r'(?!\w)'
        if re.search(pattern, text_low):
            matches.append(taxon)
    return matches

def ask_relations(abstract, candidate_taxa):
    if not candidate_taxa:
        return "()"
    
    user_prompt = f"""
Candidate taxa:
{", ".join(candidate_taxa)}

Abstract:
{abstract}
"""
    try:
        completion = client.chat.completions.create(
            model="qwen2.5:3b",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=300,
            timeout=10000,
            temperature=0
        )
        answer = completion.choices[0].message.content
        return answer.strip() if answer else "()"
    except Exception as e:
        tqdm.write(f"Error: {e}")
        return "()"

file_path = "articles_filtered.csv"
df = pd.read_csv(file_path, encoding="latin1")

relations = []
candidate_taxa_col = []

with tqdm(total=len(df), desc="Processing abstracts", dynamic_ncols=True) as pbar:
    for i, row in df.iterrows():
        abstract = str(row.get("abstract", ""))
        candidate_taxa = find_candidate_taxa(abstract, TAXA_KEYWORDS)
        candidate_taxa_col.append(", ".join(candidate_taxa) if candidate_taxa else "")
        
        response = ask_relations(abstract, candidate_taxa)
        relations.append(response)

        tqdm.write(f"Row {i+1} taxa: {candidate_taxa}")
        tqdm.write(f"Row {i+1} relations: {response}\n")
        pbar.update(1)

df["Candidate taxa"] = candidate_taxa_col
df["Extracted relations"] = relations
df.to_csv("algae_relations.csv", index=False)

Processing abstracts:   0%|          | 0/974 [00:00<?, ?it/s]

Row 1 taxa: []
Row 1 relations: ()

Row 2 taxa: ['Methanosaeta']
Row 2 relations: (Methanosaeta -> methane/biogas)

Row 3 taxa: []
Row 3 relations: ()

Row 4 taxa: ['Methanosaeta']
Row 4 relations: (Methanosaeta -> methane/biogas), ()

Row 5 taxa: ['Methanosarcina']
Row 5 relations: (Methanosarcina -> methane/biogas), (Methanosarcina -> biomass growth/productivity)

Row 6 taxa: []
Row 6 relations: ()

Row 7 taxa: ['Methanosarcina']
Row 7 relations: (Methanosarcina -> methane/biogas), ()

Row 8 taxa: ['Methanosaeta']
Row 8 relations: (Methanosaeta -> methane/biogas), ()

Row 9 taxa: []
Row 9 relations: ()

Row 10 taxa: ['Methanosaeta']
Row 10 relations: (Methanosaeta -> methane/biogas)

Row 11 taxa: []
Row 11 relations: ()

Row 12 taxa: []
Row 12 relations: ()

Row 13 taxa: ['Methanosarcina']
Row 13 relations: (Methanosarcina -> methane/biogas)

Row 14 taxa: []
Row 14 relations: ()

Row 15 taxa: []
Row 15 relations: ()

Row 16 taxa: ['Chlorella sp']
Row 16 relations: (Chlorella sp -> me

In [6]:
#Cleaning the extraction to exclude null values or for using only taxa from candidate taxa list (to avoid hallucination, I will focus only on known species
import pandas as pd
import re

TAXA_KEYWORDS = [
    "Chlamydomonas reinhardtii",
    "Chlorella sp",
    "Chlorella fusca",
    "Chlorella minutissima",
    "Chlorella protothecoides",
    "Chlorella pyrenoidosa",
    "Chlorella sorokiniana",
    "Chlorella variabilis",
    "Chlorella vulgaris",
    "Dunaliella tertiolecta",
    "Tetradesmus obliquus",
    "Nannochloropsis gaditana",
    "Nannochloropsis salina",
    "Parachlorella kessleri",
    "Scenedesmus abundans",
    "Scenedesmus sp",
    "Scenedesmus obliquus",
    "Tetraselmis sp",
    "Tetraselmis elliptica",
    "Tetraselmis suecica",
    "Methanosarcina",
    "Arthrospira maxima",
    "Arthrospira sp",
    "Phormidium sp",
    "Pseudoscillatoria coralii",
    "Acutodesmus obliquus",
    "Chlamydomonas sp",
    "Chlorella kessleri",
    "Dunaliella salina",
    "Phaeodactylum tricornutum",
    "Scenedesmus obtusiusculus",
    "Methanosarcina acetivorans",
    "Methanosaeta",
    "Chlamydomonas acidophila",
    "Neochloris oleoabundans",
    "Spirulina sp",
    "Piscicoccus intestinalis",
    "Leptolyngbya angustata",
    "Synechococcus elongatus"
]

REL_COL = "Extracted relations"  # ajuste se necessário
df = pd.read_csv("algae_relations.csv", encoding="latin1")
df[REL_COL] = df[REL_COL].fillna("").astype(str)

pair_pattern = r"\(\s*([^()]+?)\s*->\s*([^()]+?)\s*\)"

valid_outcomes = {
    "nutrient removal": "nutrient removal",
    "methane/biogas": "methane/biogas",
    "biomass growth/productivity": "biomass growth/productivity",
    "methane production": "methane/biogas",
    "biogas production": "methane/biogas",
    "methane yield": "methane/biogas",
    "biogas yield": "methane/biogas",
    "biomass growth": "biomass growth/productivity",
    "biomass productivity": "biomass growth/productivity",
    "productivity": "biomass growth/productivity",
    "growth": "biomass growth/productivity",
}

taxa_lookup = {t.lower(): t for t in TAXA_KEYWORDS}

fuzzy_taxa_map = {
    "chlorella sp.": "Chlorella sp",
    "chlorella_sp": "Chlorella sp",
    "chlorella vulgaris": "Chlorella vulgaris",
    "clorella vulgaris": "Chlorella vulgaris",
    "clorella_vulgaris": "Chlorella vulgaris",
    "chlorella_vulgaris": "Chlorella vulgaris",
    "covalva vulgaris": "Chlorella vulgaris",
    "scenedesmus sp.": "Scenedesmus sp",
    "arthrospira sp.": "Arthrospira sp",
    "tetraselmis sp.": "Tetraselmis sp",
    "chlorella sorokiniana": "Chlorella sorokiniana",
    "schuedema sp": "Scenedesmus sp",
    "a. platensis": "Arthrospira sp",
    "chlorella pyrenoidosa": "Chlorella pyrenoidosa",
}

def normalize_text(x):
    x = str(x).strip()
    x = x.replace("_", " ")
    x = re.sub(r"\s+", " ", x)
    return x

def normalize_taxon(raw):
    raw = normalize_text(raw)
    low = raw.lower()

    if low in ("", "null", "none", "na", "n/a"):
        return None

    if low in taxa_lookup:
        return taxa_lookup[low]

    if low in fuzzy_taxa_map:
        return fuzzy_taxa_map[low]

    return None

def normalize_outcome(raw):
    raw = normalize_text(raw).lower()
    if raw in ("", "null", "none", "na", "n/a"):
        return None
    return valid_outcomes.get(raw, None)

def split_taxa_field(left):
    parts = re.split(r"\s*,\s*", left)
    return [p for p in parts if p.strip()]

def clean_relations(cell):
    matches = re.findall(pair_pattern, cell)
    cleaned = []

    for left, right in matches:
        outcome = normalize_outcome(right)
        if outcome is None:
            continue

        taxa_parts = split_taxa_field(left)
        for tax in taxa_parts:
            norm_tax = normalize_taxon(tax)
            if norm_tax is not None:
                cleaned.append((norm_tax, outcome))

    cleaned = list(dict.fromkeys(cleaned))

    if not cleaned:
        return "()"

    return ", ".join(f"({tax} -> {out})" for tax, out in cleaned)

df["Extracted relations cleaned"] = df[REL_COL].apply(clean_relations)
df.to_csv("algae_relations_cleaned.csv", index=False)

In [7]:
# Remove rows with no valid relationships.
df = df[df["Extracted relations cleaned"] != "()"].copy()
df.reset_index(drop=True, inplace=True)

df.to_csv("algae_relations_cleaned.csv", index=False)

**Step 3: Combine entities with similar meanings**

In [8]:
import pandas as pd
import re
from collections import OrderedDict

df = pd.read_csv("algae_relations_cleaned.csv", encoding="latin1")

REL_COL = "Extracted relations"  # ou "Extracted entities", mas escolha um só nome
df[REL_COL] = df[REL_COL].fillna("").astype(str)

pattern = r"\(\s*([^()]+?)\s*->\s*([^()]+?)\s*\)"

taxon_map = {
    "Chlorella sp.": "Chlorella sp",
    "Scenedesmus sp.": "Scenedesmus sp",
    "Tetraselmis sp.": "Tetraselmis sp",
    "Arthrospira sp.": "Arthrospira sp",
    "Spirulina sp.": "Spirulina sp",
}

outcome_map = {
    "nutrient removal": "nutrient removal",
    "methane production": "methane/biogas",
    "biogas production": "methane/biogas",
    "methane yield": "methane/biogas",
    "biogas yield": "methane/biogas",
    "methane enhancement": "methane/biogas",
    "biomass growth": "biomass growth/productivity",
    "growth": "biomass growth/productivity",
    "biomass productivity": "biomass growth/productivity",
    "productivity": "biomass growth/productivity",
    "biomass yield": "biomass growth/productivity",
}

def normalize_taxon(x):
    x = x.strip()
    x = re.sub(r"\s+", " ", x)
    return taxon_map.get(x, x)

def normalize_outcome(x):
    x = x.strip().lower()
    x = re.sub(r"\s+", " ", x)
    return outcome_map.get(x, x)

def dedup_relations(cell):
    matches = re.findall(pattern, cell)
    if not matches:
        return "()"
    
    seen = OrderedDict()
    for left, right in matches:
        left_n = normalize_taxon(left)
        right_n = normalize_outcome(right)
        key = (left_n, right_n)
        seen[key] = None
    
    return ", ".join([f"({a} -> {b})" for a, b in seen.keys()]) if seen else "()"

df["Extracted relations (dedup)"] = df[REL_COL].apply(dedup_relations)
df.to_csv("algae_combined.csv", index=False)

**Step 4: Plot knowledge graph**

In [11]:
from pyvis.network import Network
import pandas as pd
import re
import networkx as nx

filepath = "algae_combined.csv"
df = pd.read_csv(filepath, encoding="latin1")

REL_COL = "Extracted relations cleaned"  # ou "Extracted relations (dedup)"
TITLE_COL = "title"

G = nx.DiGraph()

words_to_exclude = {"the", "this", "results", "study", "null", "none"}
pattern = r"\(\s*([^()]+?)\s*->\s*([^()]+?)\s*\)"

print("Processing relations...")

for idx, row in df.iterrows():
    value = str(row.get(REL_COL, "")).strip()
    if value in ("", "()", "nan"):
        continue

    source_title = str(row.get(TITLE_COL, f"article_{idx+1}"))[:80]
    matches = re.findall(pattern, value)

    for a_raw, b_raw in matches:
        a = a_raw.strip()
        b = b_raw.strip()

        if (
            len(a) > 2 and len(b) > 2 and
            a.lower() not in words_to_exclude and
            b.lower() not in words_to_exclude
        ):
            if not G.has_node(a):
                G.add_node(a, label=a, title=a, size=24, color="#66c2a5")
            if not G.has_node(b):
                color = "#fc8d62" if b in {
                    "nutrient removal",
                    "methane/biogas",
                    "biomass growth/productivity"
                } else "#8da0cb"
                G.add_node(b, label=b, title=b, size=20, color=color)

            G.add_edge(a, b, title=source_title)

print(f"KG: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

def search_network(graph, keywords, depth=2):
    keywords = [k.lower() for k in keywords]

    seed_nodes = {
        n for n, d in graph.nodes(data=True)
        if any(k in str(d.get("label", "")).lower() for k in keywords)
    }

    if not seed_nodes:
        return nx.DiGraph()

    visited = set(seed_nodes)
    frontier = set(seed_nodes)

    for _ in range(depth):
        new_nodes = set()
        for node in frontier:
            new_nodes.update(graph.predecessors(node))
            new_nodes.update(graph.successors(node))
        new_nodes -= visited
        visited.update(new_nodes)
        frontier = new_nodes

    return graph.subgraph(visited).copy()

keywords_algae = ["chlorella", "biogas", "nutrient removal"]
filtered_graph = search_network(G, keywords_algae, depth=2)

print(f"Filtered: {filtered_graph.number_of_nodes()} nodes, {filtered_graph.number_of_edges()} edges")

net = Network(
    height="800px",
    width="100%",
    directed=True,
    notebook=False,
    bgcolor="#222222",
    font_color="white"
)

net.from_nx(filtered_graph)

net.set_options("""
var options = {
  "nodes": {
    "shape": "dot",
    "font": {"size": 16, "color": "white"},
    "scaling": {"min": 16, "max": 32}
  },
  "edges": {
    "arrows": {"to": {"enabled": true}},
    "color": {"inherit": false},
    "smooth": false
  },
  "physics": {
    "barnesHut": {
      "gravitationalConstant": -12000,
      "centralGravity": 0.2,
      "springLength": 140,
      "springConstant": 0.03,
      "damping": 0.09
    },
    "minVelocity": 0.75
  },
  "interaction": {
    "hover": true,
    "navigationButtons": true,
    "keyboard": true
  }
}
""")

net.save_graph("algae_kg1.html")
print("algae_kg1.html saved.")

Processing relations...
KG: 40 nodes, 81 edges
Filtered: 40 nodes, 81 edges
algae_kg1.html saved.


In [15]:
import pandas as pd
import re
import json
from datetime import datetime

filepath = "algae_combined.csv"
df = pd.read_csv(filepath, encoding="latin1")

REL_COL = "Extracted relations cleaned"   # ajuste se necessário
TITLE_COL = "title"

pattern = r"\(\s*([^()]+?)\s*->\s*([^()]+?)\s*\)"

outcomes = {
    "nutrient removal",
    "methane/biogas",
    "biomass growth/productivity"
}

cyanobacteria_keywords = {
    "Arthrospira maxima", "Arthrospira sp", "Spirulina sp",
    "Phormidium sp", "Pseudoscillatoria coralii",
    "Leptolyngbya angustata", "Synechococcus elongatus"
}

archaea_keywords = {
    "Methanosarcina", "Methanosarcina acetivorans", "Methanosaeta"
}

nodes_dict = {}
edges = []
edge_seen = set()

def classify_group(name):
    if name in outcomes:
        return "Outcome"
    if name in archaea_keywords:
        return "Archaea"
    if name in cyanobacteria_keywords:
        return "Cyanobacteria"
    return "Algae"

def add_node(name):
    if name not in nodes_dict:
        group = classify_group(name)
        nodes_dict[name] = {
            "id": name,
            "label": name,
            "group": group,
            "title": f"{name} ({group})",
            "value": 1
        }
    else:
        nodes_dict[name]["value"] += 1

for idx, row in df.iterrows():
    value = str(row.get(REL_COL, "")).strip()
    if value in ("", "()", "nan"):
        continue

    source_title = str(row.get(TITLE_COL, f"article_{idx+1}"))

    matches = re.findall(pattern, value)
    for source, target in matches:
        source = source.strip()
        target = target.strip()

        if not source or not target:
            continue

        add_node(source)
        add_node(target)

        edge_key = (source, target, source_title)
        if edge_key not in edge_seen:
            edges.append({
                "from": source,
                "to": target,
                "label": "",
                "title": source_title,
                "arrows": "to"
            })
            edge_seen.add(edge_key)

nodes = list(nodes_dict.values())

today = datetime.now().strftime("%Y-%m-%d")

html = f"""
<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8"/>
  <title>Species → Outcomes Knowledge Graph</title>
  <script src="https://unpkg.com/vis-network/standalone/umd/vis-network.min.js"></script>
  <style>
    body {{
      margin: 0;
      font-family: Arial, sans-serif;
      background: #ffffff;
      color: #222;
    }}

    #legend {{
      padding: 10px 14px;
      font-size: 13px;
      border-bottom: 1px solid #eaeaea;
      background: #fafafa;
    }}

    #mynetwork {{
      height: calc(100vh - 46px);
      width: 100%;
    }}
  </style>
</head>
<body>

<div id="legend">
  <span style="margin-right:15px">
    <span style="color:#4CAF50;font-size:18px">●</span> Algae
  </span>
  <span style="margin-right:15px">
    <span style="color:#03A9F4;font-size:18px">●</span> Cyanobacteria
  </span>
  <span style="margin-right:15px">
    <span style="color:#FF9800;font-size:18px">●</span> Archaea
  </span>
  <span style="margin-right:15px">
    <span style="color:#9C27B0;font-size:18px">●</span> Outcome / Process
  </span>
</div>

<div id="mynetwork"></div>

<script>
  const nodes = new vis.DataSet({json.dumps(nodes, ensure_ascii=False)});
  const edges = new vis.DataSet({json.dumps(edges, ensure_ascii=False)});

  const container = document.getElementById("mynetwork");
  const data = {{ nodes, edges }};

  const options = {{
    nodes: {{
      shape: "dot",
      scaling: {{ min: 10, max: 40 }},
      font: {{
        size: 14,
        face: "Arial"
      }},
      borderWidth: 1.5
    }},

    edges: {{
      width: 0.8,
      color: {{
        color: "#999999",
        highlight: "#444444",
        hover: "#000000"
      }},
      smooth: {{
        enabled: true,
        type: "continuous"
      }}
    }},

    groups: {{
      Algae: {{
        color: {{ background: "#4CAF50", border: "#388E3C" }}
      }},
      Cyanobacteria: {{
        color: {{ background: "#03A9F4", border: "#0288D1" }}
      }},
      Archaea: {{
        color: {{ background: "#FF9800", border: "#F57C00" }}
      }},
      Outcome: {{
        color: {{ background: "#9C27B0", border: "#7B1FA2" }}
      }},
      Other: {{
        color: {{ background: "#FDD835", border: "#F9A825" }}
      }}
    }},

    physics: {{
      barnesHut: {{
        gravitationalConstant: -8000,
        springLength: 140,
        damping: 0.4
      }},
      stabilization: {{
        iterations: 300
      }}
    }},

    interaction: {{
      hover: true,
      zoomView: true,
      dragView: true,
      navigationButtons: true
    }}
  }};

  const network = new vis.Network(container, data, options);
</script>

</body>
</html>
"""

filename = f"kg_species_outcomes_{today}.html"

with open(filename, "w", encoding="utf-8") as f:
    f.write(html)

print(f"✔ Saved: {filename}")

✔ Saved: kg_species_outcomes_2026-05-01.html


**New KG focusing on species and outcome related to Anaerobic Digestion**

In [18]:
import pandas as pd
import re
import seaborn as sns
import matplotlib.pyplot as plt

filepath = "algae_combined.csv"   
REL_COL = "Extracted relations cleaned" 
TITLE_COL = "title"               

OUTCOMES = [
    "nutrient removal",
    "methane/biogas",
    "biomass growth/productivity"
]

label_map = {
    "nutrient removal": "nutrient\nremoval",
    "methane/biogas": "methane/\nbiogas",
    "biomass growth/productivity": "biomass growth/\nproductivity"
}

pattern = r"\(\s*([^()]+?)\s*->\s*([^()]+?)\s*\)"

# LOAD
df = pd.read_csv(filepath, encoding="latin1")
df[REL_COL] = df[REL_COL].fillna("").astype(str)

df = df[
    df[REL_COL].astype(str).str.strip().notna() &
    (df[REL_COL].astype(str).str.strip() != "") &
    (df[REL_COL].astype(str).str.strip() != "()")
].copy()

df["study_id"] = range(1, len(df) + 1)


# EXPLODE RELATIONS
records = []

for _, row in df.iterrows():
    study_id = row["study_id"]
    rel_text = str(row[REL_COL])

    matches = re.findall(pattern, rel_text)
    for species, outcome in matches:
        species = species.strip()
        outcome = outcome.strip()

        if species and outcome in OUTCOMES:
            records.append({
                "study_id": study_id,
                "species": species,
                "outcome": outcome
            })

rel_df = pd.DataFrame(records)

rel_df = rel_df.drop_duplicates(subset=["study_id", "species", "outcome"]).copy()


# OPTION A: proportion relative to total studies

total_studies = df["study_id"].nunique()

heat_total = (
    rel_df.groupby(["species", "outcome"])["study_id"]
    .nunique()
    .unstack(fill_value=0)
    .reindex(columns=OUTCOMES, fill_value=0)
)

heat_total_prop = heat_total / total_studies

species_order = heat_total_prop.sum(axis=1).sort_values(ascending=False).index
heat_total_prop = heat_total_prop.loc[species_order]


# OPTION B:proportion within species
species_totals = (
    rel_df.groupby("species")["study_id"]
    .nunique()
)

heat_species_prop = heat_total.div(species_totals, axis=0).fillna(0)
heat_species_prop = heat_species_prop.loc[species_order]

#Plot Function
def plot_heatmap(data, title, output_png):
    plot_data = data.copy()
    plot_data.columns = [label_map.get(c, c) for c in plot_data.columns]

    plt.figure(figsize=(8, max(10, len(plot_data) * 0.38)))

    ax = sns.heatmap(
        plot_data,
        cmap="YlGnBu",
        linewidths=0.5,
        linecolor="white",
        annot=True,
        fmt=".2f",
        cbar_kws={"label": "Proportion of studies"}
    )

    ax.set_title(title, fontsize=16, pad=14)
    ax.set_xlabel("Process outcome", fontsize=13)
    ax.set_ylabel("Species", fontsize=13)

    ax.set_xticklabels(ax.get_xticklabels(), rotation=0, ha="center", fontsize=11)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=11)

    plt.tight_layout()
    plt.savefig(output_png, dpi=300, bbox_inches="tight")
    plt.close()

#Savings
plot_heatmap(
    heat_total_prop,
    "Species-outcome associations relative to total studies",
    "heatmap_total_studies.png"
)

plot_heatmap(
    heat_species_prop,
    "Species-outcome associations normalized within species",
    "heatmap_within_species.png"
)

heat_total_prop.to_csv("heatmap_total_studies.csv")
heat_species_prop.to_csv("heatmap_within_species.csv")

print("Saved:")
print("- heatmap_total_studies.png")
print("- heatmap_within_species.png")
print("- heatmap_total_studies.csv")
print("- heatmap_within_species.csv")

Saved:
- heatmap_total_studies.png
- heatmap_within_species.png
- heatmap_total_studies.csv
- heatmap_within_species.csv


In [14]:
import pandas as pd
import re
import json
import math
from datetime import datetime

filepath = "algae_combined.csv"  
REL_COL = "Extracted relations cleaned"  
TITLE_COL = "title"

pattern = r"\(\s*([^()]+?)\s*->\s*([^()]+?)\s*\)"

OUTCOMES = [
    "nutrient removal",
    "methane/biogas",
    "biomass growth/productivity"
]

cyanobacteria_keywords = {
    "Arthrospira maxima", "Arthrospira sp", "Spirulina sp",
    "Phormidium sp", "Pseudoscillatoria coralii",
    "Leptolyngbya angustata", "Synechococcus elongatus"
}

archaea_keywords = {
    "Methanosarcina", "Methanosarcina acetivorans", "Methanosaeta"
}

# LOAD
df = pd.read_csv(filepath, encoding="latin1")
df[REL_COL] = df[REL_COL].fillna("").astype(str)

df = df[
    df[REL_COL].astype(str).str.strip().notna() &
    (df[REL_COL].astype(str).str.strip() != "") &
    (df[REL_COL].astype(str).str.strip() != "()")
].copy()

# HELPERS
def classify_group(name):
    if name in OUTCOMES:
        return "Outcome"
    if name in archaea_keywords:
        return "Archaea"
    if name in cyanobacteria_keywords:
        return "Cyanobacteria"
    return "Algae"

def node_color(group):
    return {
        "Algae": {"background": "#43A047", "border": "#2E7D32"},
        "Cyanobacteria": {"background": "#29B6F6", "border": "#0288D1"},
        "Archaea": {"background": "#F9A825", "border": "#F57F17"},
        "Outcome": {"background": "#9C27B0", "border": "#7B1FA2"},
        "Other": {"background": "#9E9E9E", "border": "#616161"}
    }.get(group, {"background": "#9E9E9E", "border": "#616161"})

def edge_color_by_source_group(group):
    return {
        "Algae": "#2E7D32",
        "Cyanobacteria": "#0288D1",
        "Archaea": "#D68910",
        "Outcome": "#7B1FA2",
        "Other": "#9E9E9E"
    }.get(group, "#9E9E9E")

# COUNT NODES / EDGES
node_counts = {}
edge_counts = {}
edge_titles = {}

for idx, row in df.iterrows():
    rels = str(row.get(REL_COL, "")).strip()
    title = str(row.get(TITLE_COL, f"article_{idx+1}"))

    matches = re.findall(pattern, rels)
    for source, target in matches:
        source = source.strip()
        target = target.strip()
        if not source or not target:
            continue

        node_counts[source] = node_counts.get(source, 0) + 1
        node_counts[target] = node_counts.get(target, 0) + 1

        key = (source, target)
        edge_counts[key] = edge_counts.get(key, 0) + 1
        edge_titles.setdefault(key, []).append(title)

# POSITION NODES
nodes = []
edges = []

species = [n for n in node_counts.keys() if n not in OUTCOMES]
species = sorted(species, key=lambda x: (-node_counts[x], x.lower()))

# outcomes near the center
outcome_positions = {
    "nutrient removal": (-180, 0),
    "methane/biogas": (120, -40),
    "biomass growth/productivity": (70, 170),
}

# distribute species in rings around
radius_base = 260
radius_step = 85
per_ring = 26

species_positions = {}
for i, name in enumerate(species):
    ring = i // per_ring
    pos_in_ring = i % per_ring
    n_in_ring = min(per_ring, len(species) - ring * per_ring)

    angle = (2 * math.pi * pos_in_ring / max(n_in_ring, 1)) + (ring * 0.22)
    radius = radius_base + ring * radius_step

    x = radius * math.cos(angle)
    y = radius * math.sin(angle)

    species_positions[name] = (round(x, 1), round(y, 1))

# BUILD NODES
for name, count in node_counts.items():
    group = classify_group(name)
    color = node_color(group)

    if name in OUTCOMES:
        x, y = outcome_positions[name]
        value = min(26 + count * 1.4, 44)
        font_size = 18
    else:
        x, y = species_positions[name]
        value = min(8 + count * 1.6, 28)
        font_size = 9

    nodes.append({
        "id": name,
        "label": name,
        "group": group,
        "title": f"{name}<br>Group: {group}<br>Mentions: {count}",
        "value": value,
        "x": x,
        "y": y,
        "physics": False,
        "color": color,
        "font": {"size": font_size}
    })

# BUILD EDGES
for (source, target), count in edge_counts.items():
    source_group = classify_group(source)
    color = edge_color_by_source_group(source_group)

    titles = edge_titles.get((source, target), [])
    title_preview = "<br>".join(titles[:6])
    if len(titles) > 6:
        title_preview += "<br>..."

    edges.append({
        "from": source,
        "to": target,
        "width": min(0.6 + count * 0.55, 7),
        "title": f"Count: {count}<br>{title_preview}",
        "color": {
            "color": color,
            "opacity": 0.30,
            "highlight": color,
            "hover": color
        },
        "smooth": {
            "enabled": True,
            "type": "continuous",
            "roundness": 0.22
        },
        "arrows": {
            "to": {
                "enabled": False
            }
        }
    })

# HTML
today = datetime.now().strftime("%Y-%m-%d")

html = f"""
<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8"/>
  <title>Species → Outcomes Knowledge Graph</title>
  <script src="https://unpkg.com/vis-network/standalone/umd/vis-network.min.js"></script>
  <style>
    body {{
      margin: 0;
      font-family: Arial, sans-serif;
      background: #ffffff;
      color: #222;
    }}

    #topbar {{
      height: 62px;
      box-sizing: border-box;
      display: flex;
      align-items: center;
      gap: 18px;
      padding: 10px 16px;
      border-bottom: 1px solid #e8e8e8;
      background: #ffffff;
      flex-wrap: wrap;
    }}

    .legend {{
      display: flex;
      gap: 18px;
      align-items: center;
      font-size: 13px;
      white-space: nowrap;
    }}

    .legend-item {{
      display: inline-flex;
      align-items: center;
      gap: 6px;
    }}

    .dot {{
      font-size: 18px;
      line-height: 1;
    }}

    #searchBox {{
      width: 380px;
      max-width: 45vw;
      padding: 10px 14px;
      font-size: 14px;
      border: 1px solid #9e9e9e;
      outline: none;
    }}

    #resetBtn {{
      padding: 10px 14px;
      font-size: 14px;
      border: 1px solid #7f7f7f;
      background: #f7f7f7;
      cursor: pointer;
    }}

    #mynetwork {{
      width: 100%;
      height: calc(100vh - 63px);
      background: #ffffff;
    }}
  </style>
</head>
<body>

<div id="topbar">
  <div class="legend">
    <span class="legend-item"><span class="dot" style="color:#43A047">●</span> Algae</span>
    <span class="legend-item"><span class="dot" style="color:#29B6F6">●</span> Cyanobacteria</span>
    <span class="legend-item"><span class="dot" style="color:#F9A825">●</span> Archaea</span>
    <span class="legend-item"><span class="dot" style="color:#9C27B0">●</span> Outcome / Process</span>
  </div>

  <input id="searchBox" type="text" placeholder="Search node (e.g. Chlorella)" />
  <button id="resetBtn">Reset view</button>
</div>

<div id="mynetwork"></div>

<script>
  const allNodes = {json.dumps(nodes, ensure_ascii=False)};
  const allEdges = {json.dumps(edges, ensure_ascii=False)};

  const nodes = new vis.DataSet(allNodes);
  const edges = new vis.DataSet(allEdges);

  const container = document.getElementById("mynetwork");
  const data = {{ nodes, edges }};

  const options = {{
    autoResize: true,
    physics: false,

    nodes: {{
      shape: "dot",
      scaling: {{ min: 6, max: 42 }},
      borderWidth: 1.2,
      font: {{
        face: "Arial",
        color: "#444",
        strokeWidth: 0
      }}
    }},

    edges: {{
      selectionWidth: 1.5,
      hoverWidth: 1.2
    }},

    interaction: {{
      hover: true,
      tooltipDelay: 120,
      navigationButtons: false,
      keyboard: true,
      dragNodes: true,
      dragView: true,
      zoomView: true
    }},

    layout: {{
      improvedLayout: false
    }}
  }};

  const network = new vis.Network(container, data, options);

  const initialPositions = network.getPositions();
  const initialScale = network.getScale();

  function focusNodeByText(query) {{
    if (!query) return;

    const q = query.toLowerCase().trim();
    const found = allNodes.find(n => (n.label || "").toLowerCase().includes(q));

    if (found) {{
      network.selectNodes([found.id]);
      network.focus(found.id, {{
        scale: 1.45,
        animation: {{
          duration: 700,
          easingFunction: "easeInOutQuad"
        }}
      }});
    }}
  }}

  document.getElementById("searchBox").addEventListener("keydown", function(e) {{
    if (e.key === "Enter") {{
      focusNodeByText(this.value);
    }}
  }});

  document.getElementById("resetBtn").addEventListener("click", function() {{
    network.unselectAll();
    network.fit({{
      animation: {{
        duration: 700,
        easingFunction: "easeInOutQuad"
      }}
    }});
    document.getElementById("searchBox").value = "";
  }});

  setTimeout(() => {{
    network.fit({{
      animation: false
    }});
  }}, 100);
</script>

</body>
</html>
"""

filename = f"kg_species_outcomes_visual_{today}.html"
with open(filename, "w", encoding="utf-8") as f:
    f.write(html)

print(f"✔ Saved: {filename}")

✔ Saved: kg_species_outcomes_visual_2026-05-01.html
